# BlackBox — Phase 4: Decay & Reinforcement

Phase 3 turned decay **off** on purpose, to test pure relevance ranking without a confound.
This notebook turns it **on** and tests it directly, using mem0's real documented feature:

- A soft re-rank applied only at search time -- storage is untouched
- Each memory tracks up to its last 20 access timestamps
- Recently-accessed memories get up to a `1.5×` score boost; idle ones get dampened toward `0.3×`
- Nothing is ever fully hidden -- the floor is `0.3×`, not `0×`
- Memories that existed before decay was turned on get a one-time fallback using their
  last-update timestamp, then accumulate real access history from there

Same `eng_01` data as before. No new facts needed to start -- we already have a stale one
(the 737 fact) and a fresh one (the 787 fact) sitting right there from Phase 2.


## 0. Baseline, with decay off

Same query as Phase 3, recorded one more time as a clean "before" reference. If you already
ran Phase 3 last, decay should still be off from that notebook -- this cell just confirms it.


In [1]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))

client.project.update(decay=False)

query = "what hydraulic system do I work on?"
baseline = client.search(query=query, filters={"user_id": "eng_01"}, top_k=10)

print("BASELINE (decay off):")
for r in baseline.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

BASELINE (decay off):
0.315  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
0.291  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
0.279  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
0.260  The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primary system is out of service or during a loss‑of‑pressure event.
0.237  User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
0.223  User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
0.207  User observed

## 1. Turn decay on

One call, no reindexing, no migration -- this is a project-level setting, not something
attached to individual memories.


In [3]:
client.project.update(decay=True)
print("Decay enabled for this project.")

Decay enabled for this project.


## 2. Immediate re-run -- the fallback case

Every fact we've stored so far predates decay being turned on. Per mem0's docs, memories in
that situation get a one-time fallback: their **last-update timestamp** counts as a single
past touch, so this first search after flipping the toggle won't show much of a swing yet --
real access history only starts accumulating from here.


In [13]:
immediately_after = client.search(query=query, filters={"user_id": "eng_01"}, top_k=10)

print("IMMEDIATELY AFTER enabling decay:")
for r in immediately_after.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

print("\nCompare to baseline scores above -- expect small or no change on this first call.")

IMMEDIATELY AFTER enabling decay:
0.470  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
0.434  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
0.416  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
0.388  The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primary system is out of service or during a loss‑of‑pressure event.
0.354  User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
0.333  User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
0.309  U

BASELINE (decay off):
0.315  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
0.291  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
0.279  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
0.260  The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primary system is out of service or during a loss‑of‑pressure event.
0.237  User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
0.223  User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
0.207  User observed a low hydraulic pressure gauge reading during a preflight check on a Boeing 737 on August 3, 2026

## 3. Reinforcement experiment

Decay is described as **recency-of-access**, not recency-of-creation -- a memory gets boosted
by being *retrieved*, not just by being new. We test that distinction directly: repeatedly
search in a way that surfaces the 787 fact, leave the 737 fact untouched, then compare.


In [7]:
# Repeatedly retrieve the 787 fact specifically, simulating an agent that keeps
# coming back to it in conversation. We don't touch the 737 fact at all in this loop.
for _ in range(5):
    client.search(query="787 fleet hydraulic system", filters={"user_id": "eng_01"}, top_k=1)

print("Reinforced the 787 fact 5 times via repeated search.")

Reinforced the 787 fact 5 times via repeated search.


In [9]:
reinforced = client.search(query=query, filters={"user_id": "eng_01"}, top_k=10)

print("AFTER reinforcing the 787 fact:")
for r in reinforced.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

print("\nIf reinforcement is working, the 787 fact's score should have moved up relative")
print("to the untouched 737 fact, compared to Section 2's numbers.")

AFTER reinforcing the 787 fact:
0.469  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
0.433  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
0.417  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
0.387  The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primary system is out of service or during a loss‑of‑pressure event.
0.353  User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
0.332  User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
0.308  Use

## 4. Threshold + decay interaction

The docs flag a specific edge case worth demonstrating directly: **threshold filtering
happens before the decay scaling factor is applied.** So a stale-but-relevant memory that
just cleared the threshold can come back in your results with a *final* score that's actually
below that threshold -- it stays visible, just visibly dampened.


Here’s a simple example of what is Thresold +decay interaction.
Step 1 – Threshold filtering
You set threshold = 0.7.
Two memories pass the filter:
• Memory A → score 0.8
• Memory B → score 0.75
Step 2 – Decay scaling
Memory A is very old → scaling 0.3×
Memory B was used recently → scaling 1.5×
Step 3 – Final scores
A: 0.8 × 0.3 = 0.24
B: 0.75 × 1.5 = 1.125 (clamped to 1.0)
Result
Memory B ranks higher even though its original score was lower.
Memory A can still appear, but its score may drop below the threshold because decay is only a soft ranking bias, not a filter. 

In [14]:
thresholded = client.search(query=query, filters={"user_id": "eng_01"}, threshold=0.5, top_k=10)

print("threshold=0.5, decay on:")
for r in thresholded.get("results", []):
    below = "  <-- below 0.5 despite the threshold!" if r["score"] < 0.5 else ""
    print(f"{r['score']:.3f}  {r['memory']}{below}")

threshold=0.5, decay on:
0.471  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects  <-- below 0.5 despite the threshold!
0.435  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet  <-- below 0.5 despite the threshold!
0.417  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system  <-- below 0.5 despite the threshold!
0.388  The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primary system is out of service or during a loss‑of‑pressure event.  <-- below 0.5 despite the threshold!
0.354  User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft  

If any result printed below `0.5` despite the threshold, that's the exact behavior the
FAQ describes -- not a bug, a documented consequence of filter-then-scale ordering. Worth
calling out explicitly in the demo, since it's the kind of thing that looks like a mistake
until you know why it happens.


## 5. What decay does *not* do

Two things worth stating plainly, straight from the docs, since they're easy to assume
incorrectly:


In [ ]:
# Decay never removes a memory from results outright -- 0.3x is the floor, not 0x.
# It also doesn't touch `add()` at all -- write-path timing is unaffected.
# There's no per-project tuning of aggressiveness in this release either.
print("Floor scaling factor: 0.3x (never fully hidden)")
print("Ceiling scaling factor: 1.5x")
print("Affects: search-time ranking only")
print("Does NOT affect: add(), storage, or embeddings")

## Wrap-up

What this notebook actually demonstrated, not just described:

1. **Decay is real and toggleable** -- one call, no migration, easy to show live.
2. **It rewards access, not just age** -- Section 3 is the direct test of that claim; whatever
   the numbers actually showed is what goes in the demo, not the marketing description alone.
3. **Threshold and decay interact in a specific, documented way** -- Section 4 either did or
   didn't surface a below-threshold result, and either outcome is worth explaining to the class.
4. **Decay is scoped narrowly on purpose** -- it's a ranking nudge, not a deletion mechanism,
   and it leaves `add()` untouched. That's a deliberate design choice worth contrasting with
   Phase 2's finding that storage-level conflict resolution barely happens at all: mem0 seems
   to be betting on **read-time signals (ranking, decay) over write-time cleanup (dedup,
   deletion)** as its actual strategy for staying accurate over time.

That last point is a good closing slide for the whole demo arc: Phase 1 built the loop,
Phase 2 found write-path conflicts don't get resolved, Phase 3 showed ranking can compensate
for that at read time, and Phase 4 shows decay is one more read-time lever doing the same kind
of work -- keeping old facts around but making sure they don't drown out what's current.
